# 08 — Taxonomy-Aware Diverse Panel Selection

In [1]:
from pathlib import Path
import sys

# Find the project root (parent of notebooks/) so absolute paths work
# from anywhere the notebook is launched.
PROJECT_ROOT = Path('/scratch/ctaylor/core_models_analysis')
REPORTS = PROJECT_ROOT / 'reports'
RESULTS = PROJECT_ROOT / 'results'
LOGS    = PROJECT_ROOT / 'logs'
DATA    = PROJECT_ROOT / 'data'
SCRIPTS = PROJECT_ROOT / 'scripts'
sys.path.insert(0, str(SCRIPTS))

# KBUtils NotebookSession: opens .kbcache/ alongside this notebook so
# heavy outputs (the 3,461 reaction fingerprints) can be saved/loaded
# with provenance.
from kbutillib.notebook import NotebookSession
session = NotebookSession.for_notebook(notebook_file=__file__ if '__file__' in dir() else None,
                                       project_name='core_models_analysis')
print('Cache directory:', session.kbcache_dir)
print('Notebook name :', session.notebook_name)

Cache directory: /scratch/ctaylor/core_models_analysis/notebooks/.kbcache
Notebook name : None


## What this notebook does

Picks a **new 100-model panel** from the 3,461 growers that
deliberately spans **NCBI taxonomy** in addition to network
structure.  The original panel (`scripts/select_diverse.py`,
notebook `05`, `results/selected_*`) covered `seed.reaction`
and `cpd` sets very well, but only hit **19 of ~33
grower-bearing phyla** — missing *Myxococcota* (21 growers),
*Sphingobacteriia* (20), the entire *Micrococcales* order (122
growers) and big industrial / clinical genera like
*Pseudomonas* (84 growers) and *Burkholderia* (81).

The taxonomy data was added in notebook `07` and lives at
`results/ncbi_taxonomy.csv` (3,379 / 3,461 growers resolved;
the rest are bucketed as `Unknown`).  This notebook layers
that information into a 7-pass selection algorithm:

1. **Phylum medoids** — one anchor per grower-bearing phylum.
2. **Reaction-coverage core** — greedy max-coverage on
   `seed.reaction` IDs (12 picks).
3. **Taxonomic-novelty fill** — greedy farthest-point on
   lineage rank-distance (12 picks, closes class/order gaps).
4. **Metabolite-coverage layer** — greedy max-coverage on
   cpd IDs (8 picks).
5. **Constrained extremes** — original axis extremes with
   a class-saturation guard (8 picks).
6. **Hot-taxon medoids** — picks medoids for the top missing
   grower-heavy orders / genera / families (14 picks; closes
   broad orders like *Micrococcales* (122 growers),
   *Alteromonadales* (102), *Rhodobacterales* (95) that
   Pass 3 misses because their constituent genera are
   individually small).
7. **Farthest-point Jaccard fill** — fills the rest by
   maximizing min Jaccard reaction distance.

Each pick is tagged with the pass that selected it and a
per-pass reason string.  All new artifacts use the `_tax`
suffix so the original panel is left untouched for
comparison:

- `results/selected_ids_tax.txt`
- `results/selected_models_tax.csv`
- `results/selected_models_tax.json`
- `reports/TAXONOMY_AWARE_SELECTION.md`

Implementation lives in `scripts/select_diverse_tax.py`;
this notebook calls into it and renders intermediate state.

## Caps and budget rationale

Every pass has caps. They fall into five categories. Constants live in
`scripts/select_diverse_tax.py` (lines 78-111).

### 1. The hard external constraint

- `TARGET_SIZE = 100` - matches the original `select_diverse.py` panel so the
  two can be A/B'd head-to-head. Everything else is allocated *within* this
  budget.

### 2. Per-pass pick caps (the "budget allocation" caps)

Pass 1 deterministically eats ~26 slots (one per grower-bearing phylum,
including the `Unknown` bucket - there are 26 distinct phyla among the 3,461
growers). Pass 7 mops up whatever's left. The middle five passes split the
~74 remaining slots:

| Pass | Cap (constant) | Old script | Why this magnitude |
|---|---|---|---|
| 1 phylum medoids | uncapped (= # phyla ≈ 26) | - | data-determined: one per grower-bearing phylum |
| 2 reaction coverage | `PASS2_RXN_CORE = 12` | `RXN_CORE_TARGET = 20` | down from 20 - Pass 1's phylum anchors already pull in lots of reaction diversity, so the marginal value of picks 13-20 is small |
| 3 taxonomic novelty | `PASS3_TAX_NOVELTY = 12` | (new pass) | matched to Pass 2 so "set coverage" and "taxonomic distance" get equal weight |
| 4 metabolite coverage | `PASS4_CPD_LAYER = 8` | `CPD_LAYER_TARGET = 15` | halved - reactions and metabolites correlate, so Pass 2 already drags in cpd diversity |
| 5 constrained extremes | `PASS5_EXTREMES = 8` | `EXTREMES_TARGET = 15` | 12 axes + 3 rare-reaction champions = 15 candidates; cap of 8 says "extremes deserve some seats but shouldn't dominate" |
| 6 hot-taxon medoids | `PASS6_HOT_TAXA = 14` | (new pass) | the largest discretionary bucket - closes the genus / order gaps the old script missed; capped to leave ~20 slots for Pass 7 |
| 7 Jaccard fill | `100 − len(panel)` | - | not a cap; the remainder |

Budget arithmetic: 26 + 12 + 12 + 8 + 8 + 14 = 80 budgeted picks, leaving ~20
for Pass 7. (Picks overlap between passes - a model can qualify under two
passes and only the reason strings merge - so Pass 7 in practice gets a bit
more.)

**How arbitrary are these?** Moderately. The *relative* ordering is
intentional (Pass 6 > Pass 2 reflects "closing genus gaps matters more here
than another round of greedy reaction coverage"), but you could shift any
single one by ±2-3 without materially changing the panel. They're
judgment-driven, not optimized.

### 3. Eligibility threshold

- `HOT_TAXA_MIN_GROWERS = 20` (Pass 6) - taxa with fewer than 20 growers are
  left to Pass 7's farthest-point sweep. At the 20 cutoff you get 31 hot
  orders / 45 hot families / 26 hot genera (plenty of candidates for the
  14-pick cap). Smaller → floods the candidate list with noisy small-sample
  medoids; larger → loses mid-sized clades like *Sphingomonadales* (81)
  that the algorithm *should* catch.

### 4. Saturation guard

- `EXTREMES_CLASS_MULT = 2.0` (Pass 5) - a candidate is skipped if its class
  is already at > 2× its expected panel share (expected = class's grower
  count / total growers). Gammaproteobacteria are ~55% of growers so their
  allowance is ~110% (never blocks them); the guard only bites for
  medium-frequency classes (1-10%) where stacking three axis-extremes from
  the same class would be disproportionate. A stricter `1.5` would push
  more extremes into oddball classes; a looser `3.0` would barely do
  anything.

### 5. Performance / sanity caps

- `PHYLUM_MEDOID_SAMPLE = 50` (Passes 1 & 6) - *Pseudomonadota* has 1,909
  growers → full pairwise Jaccard is ~1.8M distance computations per
  phylum. Sampling 50 reduces it to 1,225. The medoid is a robust
  statistic (you're picking the *most central* point, not estimating a
  population mean), so 50 is conservatively oversized - 20 would probably
  give the same answer.
- `EXTREMES_WALK_LIMIT = 20` (Pass 5) - when the top candidate on an axis
  fails the class-saturation guard, walk down up to 20 ranks before giving
  up and taking the unconstrained #1. The top 20 of any axis is roughly
  the top 0.6% of growers - still meaningfully "extreme." Without this
  limit the algorithm could walk arbitrarily far down and the pick would
  no longer be "extreme" in any meaningful sense.

### Bottom line

- **Principled:** `TARGET_SIZE = 100` (matches old panel);
  `PHYLUM_MEDOID_SAMPLE = 50` (real 1.8M → 1.2K compute reduction with no
  loss of medoid accuracy).
- **Quantitatively motivated but not derived:** `HOT_TAXA_MIN_GROWERS = 20`,
  `EXTREMES_WALK_LIMIT = 20`, `EXTREMES_CLASS_MULT = 2.0` - chosen for the
  right *magnitude*, not for a precise value.
- **Budget judgments (most arbitrary):** `PASS2/3/4/5/6 = 12 / 12 / 8 / 8 / 14`.
  Each could shift ±25% without changing the panel much. They encode a
  designer's allocation across "kinds of diversity"; the reductions from
  the old script's `20 / 15 / 15` reflect that the new taxonomic passes
  (1, 3, 6) already inject the diversity those greedy passes used to
  provide.


## 1. Load inputs

We need three things:

- **growers** — the 3,461 rows from `results.csv` where `grows=True`,
- **reaction & metabolite fingerprints** — `seed.reaction` and
  `cpd` sets per model (cached as `rxnsets_by_model` /
  `cpdsets_by_model` to skip the JSON re-parse on subsequent
  runs),
- **NCBI taxonomy** — `results/ncbi_taxonomy.csv` from notebook 07.

In [2]:
import json, pandas as pd, multiprocessing as mp
from collections import Counter, defaultdict
from pathlib import Path

import select_diverse_tax as sdt

growers = sdt.load_growers()
growers_by_id = {g['model_id']: g for g in growers}
grower_ids = list(growers_by_id)
print(f'{len(growers)} growers loaded')

3461 growers loaded


In [3]:
# Reaction + metabolite fingerprints.
# Notebook 04 caches 'rxnsets_by_model' (all 5,683 models).  We
# also cache the cpd sets here so this notebook is self-contained.
MODELS = DATA / 'core_models_kegg2'

def _extract(mid):
    with open(MODELS / f'{mid}.json') as f:
        m = json.load(f)
    rxns = {r['annotation'].get('seed.reaction')
            for r in m['reactions']
            if r.get('annotation', {}).get('seed.reaction')}
    cpds = {met['id'].split('_')[0] for met in m['metabolites']}
    return mid, sorted(rxns), sorted(cpds)

try:
    cached_rxns = session.cache.load('rxnsets_by_model')
    rxns_by_id = {mid: set(rxns) for mid, rxns in cached_rxns.items()
                  if mid in growers_by_id}
    print(f'rxnsets_by_model cache hit: {len(rxns_by_id)} growers')
except KeyError:
    print('rxnsets_by_model cache miss — extracting reactions from JSON ...')
    rxns_by_id = {}
    with mp.Pool(16) as pool:
        for mid, rxns, _ in pool.imap_unordered(_extract, grower_ids, chunksize=8):
            rxns_by_id[mid] = set(rxns)
    session.cache.save('rxnsets_by_model',
                       {mid: sorted(s) for mid, s in rxns_by_id.items()},
                       type_hint='json',
                       metadata={'n_models': len(rxns_by_id),
                                 'source': 'notebook 08 extraction'})

try:
    cached_cpds = session.cache.load('cpdsets_by_model')
    cpds_by_id = {mid: set(cpds) for mid, cpds in cached_cpds.items()
                  if mid in growers_by_id}
    print(f'cpdsets_by_model cache hit: {len(cpds_by_id)} growers')
except KeyError:
    print('cpdsets_by_model cache miss — extracting metabolites from JSON ...')
    cpds_by_id = {}
    with mp.Pool(16) as pool:
        for mid, _, cpds in pool.imap_unordered(_extract, grower_ids, chunksize=8):
            cpds_by_id[mid] = set(cpds)
    session.cache.save('cpdsets_by_model',
                       {mid: sorted(s) for mid, s in cpds_by_id.items()},
                       type_hint='json',
                       metadata={'n_models': len(cpds_by_id),
                                 'source': 'notebook 08 extraction'})

print(f'reactions per grower: median '
      f'{int(pd.Series([len(s) for s in rxns_by_id.values()]).median())}; '
      f'metabolites per grower: median '
      f'{int(pd.Series([len(s) for s in cpds_by_id.values()]).median())}')

rxnsets_by_model cache hit: 3461 growers


cpdsets_by_model cache hit: 3461 growers
reactions per grower: median 145; metabolites per grower: median 132


In [4]:
# NCBI taxonomy: ranks superkingdom -> species, plus organism_name.
# Unresolved rows ('missing' status) get all ranks = 'Unknown' so they
# cluster into a single bucket in Pass 1 rather than scattering.
lineage = sdt.load_taxonomy()

tax_df = pd.read_csv(RESULTS / 'ncbi_taxonomy.csv').fillna('Unknown')
tax_grower = tax_df[tax_df['assembly_accession'].isin(growers_by_id)].copy()
n_resolved = (tax_grower['status'] == 'resolved').sum()
print(f'growers with taxonomy row: {len(tax_grower)} / {len(growers)}')
print(f'  resolved: {n_resolved}; bucketed Unknown: {len(growers) - n_resolved}')
tax_grower['phylum'] = tax_grower['phylum'].replace('Unknown', 'Unknown')
# Quick phylum tally for context (top 10).
print('\nGrower phyla (top 10):')
for phy, n in tax_grower['phylum'].value_counts().head(10).items():
    print(f'  {phy:30s} {n:>5}')

growers with taxonomy row: 3461 / 3461
  resolved: 3379; bucketed Unknown: 82

Grower phyla (top 10):
  Pseudomonadota                  1909
  Actinomycetota                   539
  Bacillota                        368
  Bacteroidota                     198
  Campylobacterota                 145
  Unknown                           82
  Cyanobacteriota                   77
  Thermodesulfobacteriota           25
  Deinococcota                      23
  Myxococcota                       21


## 2. Audit the *original* panel against the grower taxonomy

Before we re-select, quantify the gaps the new algorithm needs to
close. We load `results/selected_models.json` (the original 100)
and tally how often each phylum / class / order / genus appears
in the original panel vs. the full grower set.

In [5]:
original = json.loads((RESULTS / 'selected_models.json').read_text())
original_ids = set(original['ids'])
orig_lin = pd.DataFrame([
    {'model_id': mid, **{r: lineage[mid][r] for r in sdt.LINEAGE_RANKS}}
    for mid in original['ids']
])
print(f'original panel size: {len(original_ids)}')
print(f'original panel with resolved taxonomy: '
      f'{(orig_lin["phylum"] != "Unknown").sum()}')

def rank_coverage(panel_lin, grower_lin, rank):
    panel_set = set(panel_lin[rank]) - {'Unknown'}
    grower_set = set(grower_lin[rank]) - {'Unknown'}
    return len(panel_set), len(grower_set), grower_set - panel_set

rows = []
for r in sdt.LINEAGE_RANKS:
    panel_n, grower_n, missing = rank_coverage(orig_lin, tax_grower, r)
    rows.append((r, panel_n, grower_n,
                 f'{100*panel_n/grower_n:.1f}%' if grower_n else '—',
                 len(missing)))
pd.DataFrame(rows, columns=['rank', 'panel_distinct',
                            'grower_distinct', 'coverage_pct',
                            'taxa_missing_from_panel'])

original panel size: 100
original panel with resolved taxonomy: 98


,rank,panel_distinct,grower_distinct,coverage_pct,taxa_missing_from_panel
0,superkingdom,1,1,100.0%,0
1,phylum,19,25,76.0%,6
2,class,34,55,61.8%,21
3,order,56,130,43.1%,74
4,family,77,301,25.6%,224
5,genus,87,869,10.0%,782
6,species,97,2466,3.9%,2369


In [6]:
# Which grower-bearing phyla / classes / orders does the original
# panel skip entirely? Sorted by grower count desc -> the biggest
# gaps the new algorithm should close.
def missing_table(rank, top=10):
    panel_set = set(orig_lin[rank]) - {'Unknown'}
    counts = tax_grower[rank].value_counts()
    counts = counts[~counts.index.isin(panel_set | {'Unknown'})]
    return counts.head(top).rename_axis(rank).reset_index(name='grower_count')

print('PHYLA the original panel misses (top 10 by grower count):')
print(missing_table('phylum'))
print('\nCLASSES the original panel misses (top 10):')
print(missing_table('class'))
print('\nORDERS the original panel misses (top 10):')
print(missing_table('order'))
print('\nGENERA the original panel misses (top 10):')
print(missing_table('genus'))

PHYLA the original panel misses (top 10 by grower count):
             phylum  grower_count
0       Myxococcota            21
1   Gemmatimonadota             3
2     Rhodothermota             3
3  Ignavibacteriota             1
4     Calditrichota             1
5        Balneolota             1

CLASSES the original panel misses (top 10):
              class  grower_count
0  Sphingobacteriia            20
1        Myxococcia            14
2   Bacteriovoracia             3
3    Gemmatimonadia             3
4      Rhodothermia             3
5    Acidimicrobiia             2
6       Saprospiria             1
7    Blastocatellia             1
8    Ignavibacteria             1
9       Caldilineae             1

ORDERS the original panel misses (top 10):
                order  grower_count
0       Micrococcales           122
1     Rhodobacterales            95
2    Kitasatosporales            70
3         Vibrionales            59
4   Pseudonocardiales            36
5   Oceanospirillales    

## 3. Pass 1 — phylum medoids

For every grower-bearing phylum, pick the model whose reaction
fingerprint is the most central (minimum sum-Jaccard distance to
its phylum-mates).  Phyla with only 1–2 growers get their lone
member; phyla with > 50 growers are subsampled to 50
deterministically (stride sampling on sorted IDs) to keep the
all-pairs Jaccard tractable.

This guarantees that **every phylum present in the growers** has
at least one panel anchor — closing the largest original gap.

In [7]:
p1 = sdt.pass1_phylum_medoids(grower_ids, rxns_by_id, lineage)
print(f'Pass 1 picked {len(p1)} medoids (one per grower-bearing phylum)')
# Show first 10
print('\n# | model_id        | phylum                  | sum_jaccard')
for i, (mid, phy, sj) in enumerate(p1[:15], 1):
    print(f'{i:2d} | {mid:15s} | {phy:24s} | {sj}')

Pass 1 picked 26 medoids (one per grower-bearing phylum)



# | model_id        | phylum                  | sum_jaccard
 1 | GCF_000746525.1 | Pseudomonadota           | 11.409
 2 | GCF_000024785.1 | Actinomycetota           | 10.912
 3 | GCF_004006435.1 | Bacillota                | 12.441
 4 | GCF_003076455.1 | Bacteroidota             | 11.569
 5 | GCF_000304375.1 | Campylobacterota         | 10.497
 6 | GCF_000016865.1 | Unknown                  | 13.373
 7 | GCF_000317045.1 | Cyanobacteriota          | 8.379
 8 | GCF_001278055.1 | Thermodesulfobacteriota  | 6.84
 9 | GCF_009017495.1 | Deinococcota             | 4.052
10 | GCF_000280925.3 | Myxococcota              | 3.925
11 | GCF_009720525.1 | Planctomycetota          | 3.317
12 | GCF_000178955.2 | Acidobacteriota          | 2.377
13 | GCF_000317895.1 | Bdellovibrionota         | 0.981
14 | GCF_001747405.1 | Chlorobiota              | 1.175
15 | GCF_000018865.1 | Chloroflexota            | 0.803


## 4. Pass 2 — reaction-coverage core

Standard greedy max-coverage on `seed.reaction` IDs, **seeded
with the Pass 1 picks**.  Each new pick adds the most
previously-uncovered reactions.  Capped at 12 — fewer than the
original's 20 because Pass 1 has already pulled in a lot of
reaction diversity through the phylum anchors.

In [8]:
picked_after_p1 = {mid for mid, _, _ in p1}
p2, covered_after_p2 = sdt.greedy_max_coverage(
    grower_ids, rxns_by_id, sdt.PASS2_RXN_CORE, picked_after_p1)
print(f'Pass 2 picked {len(p2)} models')
for i, (mid, gain) in enumerate(p2, 1):
    lin = lineage[mid]
    print(f'{i:2d} | {mid:15s} | +{gain:>3d} novel rxns | '
          f'{lin["phylum"]:18s} {lin["class"]:18s} {lin["genus"]}')

Pass 2 picked 8 models
 1 | GCF_000018625.1 | +  5 novel rxns | Pseudomonadota     Gammaproteobacteria Salmonella
 2 | GCF_000025265.1 | +  3 novel rxns | Actinomycetota     Thermoleophilia    Conexibacter
 3 | GCF_001266795.1 | +  2 novel rxns | Pseudomonadota     Gammaproteobacteria Marinobacter
 4 | GCF_009688965.1 | +  2 novel rxns | Thermodesulfobacteriota Desulfobacteria    Desulfosarcina
 5 | GCF_000008525.1 | +  1 novel rxns | Campylobacterota   Epsilonproteobacteria Helicobacter
 6 | GCF_000015745.1 | +  1 novel rxns | Bacillota          Bacilli            Geobacillus
 7 | GCF_000016745.1 | +  1 novel rxns | Thermodesulfobacteriota Desulfuromonadia   Geotalea
 8 | GCF_000756615.1 | +  1 novel rxns | Bacillota          Bacilli            Paenibacillus


## 5. Pass 3 — taxonomic-novelty fill

Greedy **farthest-point on lineage rank-distance**:
7 = different superkingdom, 6 = different phylum, …, 1 = different
species, 0 = identical lineage.  Tie-break with Jaccard reaction
distance.  This is the pass that explicitly fills holes at the
class / order / genus levels — for example, Pass 1 anchors
Bacteroidota but doesn't necessarily hit *Sphingobacteriia*; Pass
3 will pick the most-distant unrepresented class first.

In [9]:
picked_after_p2 = picked_after_p1 | {mid for mid, _ in p2}
p3 = sdt.pass3_taxonomic_novelty(grower_ids, rxns_by_id, lineage,
                                 sdt.PASS3_TAX_NOVELTY,
                                 picked_after_p2)
print(f'Pass 3 picked {len(p3)} models')
print('# | model_id        | rank-d | tiebreak Jaccard | lineage (phylum/class/order/genus)')
for i, (mid, rd, tj) in enumerate(p3, 1):
    lin = lineage[mid]
    print(f'{i:2d} | {mid:15s} | {rd:>5d}  | {tj:>14.3f}   | '
          f'{lin["phylum"]}/{lin["class"]}/{lin["order"]}/{lin["genus"]}')

Pass 3 picked 12 models
# | model_id        | rank-d | tiebreak Jaccard | lineage (phylum/class/order/genus)
 1 | GCF_000015445.1 |     5  |          0.557   | Pseudomonadota/Alphaproteobacteria/Hyphomicrobiales/Bartonella
 2 | GCF_001189515.2 |     5  |          0.500   | Actinomycetota/Coriobacteriia/Coriobacteriales/Olsenella
 3 | GCF_002082765.1 |     5  |          0.494   | Bacillota/Negativicutes/Veillonellales/Veillonella
 4 | GCF_001688905.2 |     5  |          0.471   | Pseudomonadota/Betaproteobacteria/Burkholderiales/Unknown
 5 | GCF_000194135.1 |     5  |          0.461   | Campylobacterota/Desulfurellia/Desulfurellales/Hippea
 6 | GCF_000507245.1 |     5  |          0.461   | Spirochaetota/Spirochaetia/Spirochaetales/Salinispira
 7 | GCF_001688725.2 |     5  |          0.456   | Bacteroidota/Bacteroidia/Bacteroidales/Bacteroides
 8 | GCF_000021485.1 |     5  |          0.429   | Pseudomonadota/Acidithiobacillia/Acidithiobacillales/Acidithiobacillus
 9 | GCF_002005145.1 |  

## 6. Pass 4 — metabolite-coverage layer

Same greedy idea as Pass 2, but on `cpd` IDs and seeded with
every pick so far.  Catches transporter / cofactor diversity
that the reaction-set passes miss.

In [10]:
picked_after_p3 = picked_after_p2 | {mid for mid, _, _ in p3}
p4, _ = sdt.greedy_max_coverage(grower_ids, cpds_by_id,
                                sdt.PASS4_CPD_LAYER, picked_after_p3)
print(f'Pass 4 picked {len(p4)} models')
for i, (mid, gain) in enumerate(p4, 1):
    lin = lineage[mid]
    print(f'{i:2d} | {mid:15s} | +{gain:>3d} novel cpds | '
          f'{lin["phylum"]:18s} {lin["genus"]}')

Pass 4 picked 0 models


## 7. Pass 5 — constrained extremes

Original 12 axis extremes (min/max reactions, metabolites, genes,
flux, exchanges, fingerprint sizes) + 3 rare-reaction champions,
but with a **class over-representation guard**: if an axis
winner's class already has > 2× its expected panel share, walk
down the sorted ranking up to 20 candidates to find one whose
class is not yet saturated.  Cap at 8 new picks (`PASS5_EXTREMES`).

In [11]:
# Build the universe-prevalence / rare set before calling Pass 5.
prevalence = Counter()
for s in rxns_by_id.values():
    prevalence.update(s)
rare_set = {r for r, c in prevalence.items()
            if c / len(rxns_by_id) < 0.01}
print(f'rare seed.reactions (<1% of growers): {len(rare_set)}')

picked_after_p4 = picked_after_p3 | {mid for mid, _ in p4}
p5 = sdt.pass5_constrained_extremes(growers_by_id, rxns_by_id,
                                    cpds_by_id, rare_set, lineage,
                                    picked_after_p4,
                                    cap=sdt.PASS5_EXTREMES)
print(f'\nPass 5 picked {sum(1 for mid, _ in p5 if mid not in picked_after_p4)} new models '
      f'(some axes may resolve to a model already picked)')
for mid, reason in p5:
    lin = lineage[mid]
    tag = '(NEW)' if mid not in picked_after_p4 else '(dup)'
    print(f'  {tag} {mid:15s} | {lin["class"]:22s} | {reason}')

rare seed.reactions (<1% of growers): 8

Pass 5 picked 8 new models (some axes may resolve to a model already picked)
  (NEW) GCF_003261575.2 | Gammaproteobacteria    | extreme: largest by reactions
  (NEW) GCF_000283635.1 | Bacilli                | extreme: smallest by reactions
  (NEW) GCF_000632475.1 | Alphaproteobacteria    | extreme: largest by metabolites
  (NEW) GCF_000014405.1 | Bacilli                | extreme: smallest by metabolites
  (NEW) GCF_000599545.1 | Actinomycetes          | extreme: largest by genes
  (NEW) GCF_000014425.1 | Bacilli                | extreme: smallest by genes
  (NEW) GCF_000021045.1 | Gammaproteobacteria    | extreme: highest growth flux
  (NEW) GCF_000195855.1 | Actinomycetes          | extreme: lowest growth flux (still growing)


## 8. Pass 6 — hot-taxon medoids

Closes within-phylum gaps that Pass 3 misses because it
prioritizes higher-rank distance.  For each rank in
`['order', 'genus', 'family']` (broadest first), enumerate
grower-bearing taxa with ≥20 growers; for each taxon NOT yet
in the panel, add its medoid.  Cap 14 picks.

This is what brings in *Micrococcales* (122 growers),
*Alteromonadales* (102), *Rhodobacterales* (95),
*Vibrionales*, *Sphingomonadales* — broad orders whose
phylum is anchored but whose specific order had no
representative.  Many also bring in big industrial / clinical
genera (*Pseudomonas*, *Streptomyces*, *Vibrio*, *Salmonella*,
*Acinetobacter*) as a side-effect of the order medoid.

In [12]:
picked_after_p5 = picked_after_p4 | {mid for mid, _ in p5}
p6 = sdt.pass6_hot_taxon_medoids(grower_ids, rxns_by_id, lineage,
                                  sdt.PASS6_HOT_TAXA,
                                  picked_after_p5)
print(f'Pass 6 picked {len(p6)} hot-taxon medoids')
for i, (mid, rank, taxon, nm) in enumerate(p6, 1):
    lin = lineage[mid]
    print(f'{i:2d} | {mid:15s} | {rank:7s}={taxon:22s} | '
          f'{nm:>4d} growers | {lin["phylum"]}/{lin["class"]}')

Pass 6 picked 14 hot-taxon medoids
 1 | GCF_003991855.1 | order  =Micrococcales          |  122 growers | Actinomycetota/Actinomycetes
 2 | GCF_001562115.1 | order  =Alteromonadales        |  102 growers | Pseudomonadota/Gammaproteobacteria
 3 | GCF_006351965.1 | order  =Rhodobacterales        |   95 growers | Pseudomonadota/Alphaproteobacteria
 4 | GCF_000816885.1 | order  =Lysobacterales         |   86 growers | Pseudomonadota/Gammaproteobacteria
 5 | GCF_000013325.1 | order  =Sphingomonadales       |   81 growers | Pseudomonadota/Alphaproteobacteria
 6 | GCF_001278075.1 | order  =Kitasatosporales       |   70 growers | Actinomycetota/Actinomycetes
 7 | GCF_000018445.1 | order  =Moraxellales           |   62 growers | Pseudomonadota/Gammaproteobacteria
 8 | GCF_000039765.1 | order  =Vibrionales            |   59 growers | Pseudomonadota/Gammaproteobacteria
 9 | GCF_000016105.1 | order  =Thiotrichales          |   44 growers | Pseudomonadota/Gammaproteobacteria
10 | GCF_000010945.1 | 

## 9. Pass 7 — farthest-point Jaccard fill

Fill the remaining slots up to 100 by repeatedly adding the
grower whose reaction-set Jaccard distance to the panel is
maximal.  Taxonomy-blind, on purpose — this pass is a pure
diversity completion layer and lets the algorithm find odd
outliers that didn't qualify under any other axis.

In [13]:
picked_after_p6 = picked_after_p5 | {mid for mid, _, _, _ in p6}
remaining = 100 - len(picked_after_p6)
print(f'panel size before Pass 7: {len(picked_after_p6)}; '
      f'fill {remaining} more')

p7 = sdt.pass7_farthest_point(grower_ids, rxns_by_id,
                              remaining, picked_after_p6)
for i, (mid, dist) in enumerate(p7[:15], 1):
    lin = lineage[mid]
    print(f'{i:2d} | {mid:15s} | min-Jaccard {dist:.3f} | '
          f'{lin["phylum"]}/{lin["class"]}/{lin["genus"]}')
if len(p7) > 15:
    print(f'... {len(p7) - 15} more picks')

panel size before Pass 7: 68; fill 32 more


 1 | GCF_000270285.1 | min-Jaccard 0.413 | Actinomycetota/Coriobacteriia/Eggerthella
 2 | GCF_002127965.1 | min-Jaccard 0.395 | Pseudomonadota/Betaproteobacteria/Oxalobacter
 3 | GCF_003073475.1 | min-Jaccard 0.383 | Actinomycetota/Actinomycetes/Actinobaculum
 4 | GCF_900186975.1 | min-Jaccard 0.382 | Actinomycetota/Actinomycetes/Cutibacterium
 5 | GCF_008805035.1 | min-Jaccard 0.379 | Pseudomonadota/Betaproteobacteria/Eikenella
 6 | GCF_000747315.1 | min-Jaccard 0.370 | Actinomycetota/Actinomycetes/Corynebacterium
 7 | GCF_001888165.1 | min-Jaccard 0.370 | Thermodesulfobacteriota/Desulfuromonadia/Syntrophotalea
 8 | GCF_001042635.1 | min-Jaccard 0.369 | Actinomycetota/Actinomycetes/Bifidobacterium
 9 | GCF_000284095.1 | min-Jaccard 0.364 | Bacillota/Negativicutes/Pseudoselenomonas
10 | GCF_009649955.1 | min-Jaccard 0.355 | Bacillota/Clostridia/Heliorestis
11 | GCF_000828835.1 | min-Jaccard 0.353 | Pseudomonadota/Gammaproteobacteria/Thioploca
12 | GCF_000298115.2 | min-Jaccard 0.353 | 

## 10. Run the full selection end-to-end & display the panel

Re-run the algorithm through `select_diverse_tax.run_selection`
so the per-model records carry the merged `reason` strings,
lineage columns, fingerprint sizes, and rare-reaction counts.

In [14]:
selection, coverage, tax_cov, rare_set, prevalence, total_cpds = \
    sdt.run_selection(growers, rxns_by_id, cpds_by_id, lineage)

pass_dist = Counter(s['pass_origin'] for s in selection)
print('Final pass distribution:')
for k in sorted(pass_dist):
    print(f'  pass {k}: {pass_dist[k]} picks')
print(f'\nReaction coverage:    {coverage["reactions_covered"]} / '
      f'{coverage["reactions_universe"]}')
print(f'Metabolite coverage:  {coverage["metabolites_covered"]} / '
      f'{coverage["metabolites_universe"]}')

Pass 1: phylum medoids


  picked 26 medoids (one per grower-bearing phylum); panel size now 26
Pass 2: greedy reaction max-coverage (12 picks)


  added 8 picks (panel size 34)
Pass 3: taxonomic-novelty greedy (12 picks)


  added 12 picks (panel size 46)
Pass 4: greedy metabolite max-coverage (8 picks)
  added 0 picks (panel size 46)
Pass 5: constrained extremes (cap 8)
  added 8 new extreme picks (panel size 54)
Pass 6: hot-taxon medoids (cap 14)


  added 14 picks (panel size 68)
Pass 7: farthest-point Jaccard fill (32 more picks)


  added 32 picks (final panel size 100)
Final pass distribution:
  pass 1: 26 picks
  pass 2: 8 picks
  pass 3: 12 picks
  pass 5: 8 picks
  pass 6: 14 picks
  pass 7: 32 picks

Reaction coverage:    239 / 239
Metabolite coverage:  181 / 181


In [15]:
# First 25 rows for inspection.
panel_df = pd.DataFrame(selection)
panel_df[['model_id', 'pass_origin', 'organism_name', 'phylum',
          'class', 'order', 'genus', 'n_reactions',
          'n_rare_rxns', 'growth_flux']].head(25)

,model_id,pass_origin,organism_name,phylum,class,order,genus,n_reactions,n_rare_rxns,growth_flux
0,GCF_000746525.1,1,Pseudomonas alkylphenolica,Pseudomonadota,Gammaproteobacteria,Pseudomonadales,Pseudomonas,194,0,77.854
1,GCF_000024785.1,1,Gordonia bronchialis DSM 43247,Actinomycetota,Actinomycetes,Mycobacteriales,Gordonia,182,0,56.142
2,GCF_004006435.1,1,Bacillus halotolerans,Bacillota,Bacilli,Caryophanales,Bacillus,187,0,60.951
3,GCF_003076455.1,1,Flavobacterium faecale,Bacteroidota,Flavobacteriia,Flavobacteriales,Flavobacterium,162,0,43.102
4,GCF_000304375.1,1,Campylobacter jejuni subsp. jejuni NCTC 11168-...,Campylobacterota,Epsilonproteobacteria,Campylobacterales,Campylobacter,130,0,18.599
5,GCF_000016865.1,1,,Unknown,Unknown,Unknown,Unknown,194,0,62.063
6,GCF_000317045.1,1,Geitlerinema sp. PCC 7407,Cyanobacteriota,Cyanophyceae,Geitlerinematales,Geitlerinema,149,0,30.826
7,GCF_001278055.1,1,Desulfuromonas soudanensis,Thermodesulfobacteriota,Desulfuromonadia,Desulfuromonadales,Desulfuromonas,167,0,49.465
8,GCF_009017495.1,1,Deinococcus sp. AJ005,Deinococcota,Deinococci,Deinococcales,Deinococcus,162,0,13.363
9,GCF_000280925.3,1,Pseudomyxococcus hansupus,Myxococcota,Myxococcia,Myxococcales,Pseudomyxococcus,165,1,46.251


## 11. Write outputs & cache panel for downstream notebooks

Write the new panel to `results/selected_models_tax.{csv,json}`,
`results/selected_ids_tax.txt`, and
`reports/TAXONOMY_AWARE_SELECTION.md`.  The original
`selected_models.*` artifacts are LEFT IN PLACE so both panels
coexist.

In [16]:
sdt.write_outputs(selection, coverage, tax_cov, suffix='_tax')

# Cache as a typed object for downstream notebooks
session.cache.save('tax_aware_panel_v1',
                   {'ids': [s['model_id'] for s in selection],
                    'coverage': coverage,
                    'taxonomy_coverage': tax_cov,
                    'pass_distribution': dict(pass_dist)},
                   type_hint='json',
                   metadata={'n_models': len(selection),
                             'algorithm': 'tax_aware_v1',
                             'distinct_phyla': tax_cov['phylum_covered'],
                             'distinct_classes': tax_cov['class_covered'],
                             'role': 'taxonomy-aware descriptive test panel'})
print('cached tax_aware_panel_v1')

wrote /scratch/ctaylor/core_models_analysis/results/selected_models_tax.csv
wrote /scratch/ctaylor/core_models_analysis/results/selected_ids_tax.txt
wrote /scratch/ctaylor/core_models_analysis/results/selected_models_tax.json
cached tax_aware_panel_v1


## 12. Side-by-side comparison with the original panel

How does the new panel stack up against `results/selected_*.json`
on the four axes that matter:

1. **Taxonomic coverage** at each rank (phylum → genus)
2. **Set coverage** (reactions, metabolites)
3. **Representativeness** (Jaccard min-distance from every
   non-panel grower to its nearest panel member)
4. **Per-phylum / per-class panel counts** (visual)

In [17]:
new = json.loads((RESULTS / 'selected_models_tax.json').read_text())
# 1. Taxonomic coverage
new_lin = pd.DataFrame([
    {'model_id': r['model_id'],
     **{rank: r[rank] for rank in sdt.LINEAGE_RANKS}}
    for r in new['records']
])

cmp_rows = []
for r in sdt.LINEAGE_RANKS:
    orig_set = set(orig_lin[r]) - {'Unknown'}
    new_set = set(new_lin[r]) - {'Unknown'}
    grower_set = set(tax_grower[r]) - {'Unknown'}
    cmp_rows.append({
        'rank': r,
        'grower_universe': len(grower_set),
        'original_panel': len(orig_set),
        'new_panel': len(new_set),
        'delta': len(new_set) - len(orig_set),
    })
tax_cmp = pd.DataFrame(cmp_rows)
print('TAXONOMIC COVERAGE:')
print(tax_cmp.to_string(index=False))

TAXONOMIC COVERAGE:
        rank  grower_universe  original_panel  new_panel  delta
superkingdom                1               1          1      0
      phylum               25              19         25      6
       class               55              34         41      7
       order              130              56         69     13
      family              301              77         89     12
       genus              869              87         96      9
     species             2466              97         99      2


In [18]:
# 2. Set coverage comparison
set_rows = [
    ('reactions_covered',
     original['coverage']['reactions_covered'],
     new['coverage']['reactions_covered'],
     original['coverage']['reactions_universe']),
    ('metabolites_covered',
     original['coverage']['metabolites_covered'],
     new['coverage']['metabolites_covered'],
     original['coverage']['metabolites_universe']),
]
set_cmp = pd.DataFrame(set_rows, columns=['metric', 'original', 'new',
                                          'universe'])
set_cmp['orig_pct'] = (100 * set_cmp['original'] / set_cmp['universe']).round(1)
set_cmp['new_pct'] = (100 * set_cmp['new'] / set_cmp['universe']).round(1)
set_cmp['delta_pct'] = (set_cmp['new_pct'] - set_cmp['orig_pct']).round(1)
print('SET COVERAGE:')
print(set_cmp.to_string(index=False))

SET COVERAGE:
             metric  original  new  universe  orig_pct  new_pct  delta_pct
  reactions_covered       239  239       239     100.0    100.0        0.0
metabolites_covered       181  181       181     100.0    100.0        0.0


In [19]:
# 3. Representativeness: min Jaccard distance from non-panel grower
# to nearest panel member, for BOTH panels.
import statistics

def jacc(a, b):
    if not a and not b: return 0.0
    return 1.0 - len(a & b) / len(a | b)

def panel_min_dists(panel_ids):
    sel_sets = [rxns_by_id[i] for i in panel_ids]
    non_panel = [m for m in rxns_by_id if m not in set(panel_ids)]
    dists = [min(jacc(rxns_by_id[m], s) for s in sel_sets)
             for m in non_panel]
    return dists

dists_orig = panel_min_dists(original['ids'])
dists_new  = panel_min_dists(new['ids'])

def stats(ds):
    ds = sorted(ds)
    n = len(ds)
    return {
        'n_non_panel': n,
        'median': statistics.median(ds),
        'mean': sum(ds) / n,
        'p90': ds[int(0.9 * (n - 1))],
        'max': ds[-1],
    }

jstats = {'original': stats(dists_orig), 'new': stats(dists_new)}
rep_cmp = pd.DataFrame(jstats).T
print('REPRESENTATIVENESS (min Jaccard distance to nearest panel member):')
print(rep_cmp.round(3).to_string())

REPRESENTATIVENESS (min Jaccard distance to nearest panel member):
          n_non_panel  median   mean    p90    max
original       3361.0   0.192  0.181  0.255  0.304
new            3361.0   0.149  0.147  0.237  0.309


In [20]:
# 4a. Per-phylum panel counts (top 20 phyla by grower count).
top_phyla = (tax_grower['phylum'].value_counts()
             .drop('Unknown', errors='ignore')
             .head(20).index.tolist())

def phylum_counts(panel_lin, phyla):
    c = panel_lin['phylum'].value_counts()
    return [int(c.get(p, 0)) for p in phyla]

per_phylum = pd.DataFrame({
    'grower_count': [int(tax_grower['phylum'].value_counts().get(p, 0))
                     for p in top_phyla],
    'original_panel': phylum_counts(orig_lin, top_phyla),
    'new_panel': phylum_counts(new_lin, top_phyla),
}, index=top_phyla)
per_phylum['delta'] = per_phylum['new_panel'] - per_phylum['original_panel']
print('Per-phylum panel composition (top 20 grower phyla):')
print(per_phylum.to_string())

Per-phylum panel composition (top 20 grower phyla):
                         grower_count  original_panel  new_panel  delta
Pseudomonadota                   1909              32         33      1
Actinomycetota                    539              11         15      4
Bacillota                         368              18         14     -4
Bacteroidota                      198               8          5     -3
Campylobacterota                  145               6          3     -3
Cyanobacteriota                    77               1          2      1
Thermodesulfobacteriota            25               6          5     -1
Deinococcota                       23               1          1      0
Myxococcota                        21               0          1      1
Planctomycetota                    15               2          2      0
Acidobacteriota                    10               3          2     -1
Bdellovibrionota                    9               1          1      0
Chlorobiota 

In [21]:
# 4b. Per-class panel counts (top 15 classes by grower count).
top_classes = (tax_grower['class'].value_counts()
               .drop('Unknown', errors='ignore')
               .head(15).index.tolist())

def class_counts(panel_lin, classes):
    c = panel_lin['class'].value_counts()
    return [int(c.get(cls, 0)) for cls in classes]

per_class = pd.DataFrame({
    'grower_count': [int(tax_grower['class'].value_counts().get(c, 0))
                     for c in top_classes],
    'original_panel': class_counts(orig_lin, top_classes),
    'new_panel': class_counts(new_lin, top_classes),
}, index=top_classes)
per_class['delta'] = per_class['new_panel'] - per_class['original_panel']
print('Per-class panel composition (top 15 grower classes):')
print(per_class.to_string())

Per-class panel composition (top 15 grower classes):
                       grower_count  original_panel  new_panel  delta
Gammaproteobacteria            1053              14         19      5
Actinomycetes                   531               8         12      4
Alphaproteobacteria             492               7          5     -2
Betaproteobacteria              356              10          8     -2
Bacilli                         348              11          9     -2
Epsilonproteobacteria           144               5          2     -3
Flavobacteriia                  115               3          2     -1
Cyanophyceae                     77               1          2      1
Cytophagia                       34               2          1     -1
Deinococci                       23               1          1      0
Sphingobacteriia                 20               0          0      0
Chitinophagia                    16               1          1      0
Desulfuromonadia                 15  

In [22]:
# 4c. Jaccard min-distance histogram (ASCII).
def bucketize(dists, edges):
    buckets = [0] * (len(edges) - 1)
    for d in dists:
        for i in range(len(edges) - 1):
            if edges[i] <= d < edges[i + 1]:
                buckets[i] += 1
                break
        else:
            buckets[-1] += 1
    return buckets

edges = [i / 20 for i in range(11)]  # 0.00, 0.05, 0.10, ..., 0.50
b_orig = bucketize(dists_orig, edges)
b_new  = bucketize(dists_new,  edges)

hist = pd.DataFrame({
    'jaccard_bin': [f'[{edges[i]:.2f}, {edges[i+1]:.2f})'
                    for i in range(len(edges) - 1)],
    'original': b_orig,
    'new':      b_new,
    'delta':    [n - o for o, n in zip(b_orig, b_new)],
})
print('Min-Jaccard-distance distribution '
      '(every non-panel grower\'s distance to its nearest panel member):')
print(hist.to_string(index=False))
print(f'\nOriginal panel worst-case: {max(dists_orig):.3f}')
print(f'New panel worst-case:      {max(dists_new):.3f}')

Min-Jaccard-distance distribution (every non-panel grower's distance to its nearest panel member):
 jaccard_bin  original  new  delta
[0.00, 0.05)       116  272    156
[0.05, 0.10)       302  645    343
[0.10, 0.15)       545  773    228
[0.15, 0.20)       922  889    -33
[0.20, 0.25)      1065  569   -496
[0.25, 0.30)       398  204   -194
[0.30, 0.35)        13    9     -4
[0.35, 0.40)         0    0      0
[0.40, 0.45)         0    0      0
[0.45, 0.50)         0    0      0

Original panel worst-case: 0.304
New panel worst-case:      0.309


## 13. Taxonomic gap closure

Which phyla / classes / orders / genera that were **absent
from the original panel** are now covered by the new panel?
And which big-population gaps remain?

In [23]:
def gap_closure(rank, top=15):
    orig_set = set(orig_lin[rank]) - {'Unknown'}
    new_set  = set(new_lin[rank])  - {'Unknown'}
    grower_counts = tax_grower[rank].value_counts()
    closed = sorted(new_set - orig_set,
                    key=lambda t: -grower_counts.get(t, 0))
    still_missing = sorted(set(grower_counts.index) - new_set - {'Unknown'},
                           key=lambda t: -grower_counts.get(t, 0))
    return [(t, int(grower_counts.get(t, 0))) for t in closed][:top], \
           [(t, int(grower_counts.get(t, 0))) for t in still_missing][:top]

for r in ['phylum', 'class', 'order', 'genus']:
    closed, missing = gap_closure(r)
    print(f'\n=== {r.upper()} ===')
    print('NEWLY COVERED by taxonomy-aware panel:')
    for t, n in closed:
        print(f'  + {t:30s} ({n} growers)')
    if not closed:
        print('  (none — same set of taxa as original at this rank)')
    print('STILL MISSING (top remaining gaps):')
    for t, n in missing[:5]:
        print(f'  - {t:30s} ({n} growers)')


=== PHYLUM ===
NEWLY COVERED by taxonomy-aware panel:
  + Myxococcota                    (21 growers)
  + Rhodothermota                  (3 growers)
  + Gemmatimonadota                (3 growers)
  + Calditrichota                  (1 growers)
  + Balneolota                     (1 growers)
  + Ignavibacteriota               (1 growers)
STILL MISSING (top remaining gaps):

=== CLASS ===
NEWLY COVERED by taxonomy-aware panel:
  + Myxococcia                     (14 growers)
  + Gemmatimonadia                 (3 growers)
  + Rhodothermia                   (3 growers)
  + Calditrichia                   (1 growers)
  + Ignavibacteria                 (1 growers)
  + Blastocatellia                 (1 growers)
  + Balneolia                      (1 growers)
  + Phycisphaerae                  (1 growers)
STILL MISSING (top remaining gaps):
  - Sphingobacteriia               (20 growers)
  - Bacteriovoracia                (3 growers)
  - Acidimicrobiia                 (2 growers)
  - Syntrophobact

In [24]:
# Persist Jaccard stats + gap closures into the report file.
gap_for_report = {}
for r in ['phylum', 'class', 'order', 'genus']:
    closed, _ = gap_closure(r, top=15)
    gap_for_report[r] = closed

sdt.write_report(selection, coverage, tax_cov,
                 jaccard_stats={'original': stats(dists_orig),
                                'new':      stats(dists_new)},
                 gap_closures=gap_for_report)

wrote /scratch/ctaylor/core_models_analysis/reports/TAXONOMY_AWARE_SELECTION.md


## 14. Programmatic access

In [25]:
from pathlib import Path
print('From a fresh script:')
print('  ids = Path("/scratch/ctaylor/core_models_analysis/results/selected_ids_tax.txt").read_text().split()')
print()
print('Or, in a notebook that opens its own NotebookSession:')
print('  panel = session.cache.load("tax_aware_panel_v1")')
print('  ids = panel["ids"]')
print('  print(panel["taxonomy_coverage"])')

From a fresh script:
  ids = Path("/scratch/ctaylor/core_models_analysis/results/selected_ids_tax.txt").read_text().split()

Or, in a notebook that opens its own NotebookSession:
  panel = session.cache.load("tax_aware_panel_v1")
  ids = panel["ids"]
  print(panel["taxonomy_coverage"])


---
## Report: `reports/TAXONOMY_AWARE_SELECTION.md`

# Taxonomy-Aware Diverse Panel (100 models)

A taxonomy-aware re-selection of 100 representative growers, derived from the same 3,461 growers used by `scripts/select_diverse.py`. The original panel covered network sets well but missed entire phyla and big industrial/clinical genera; this panel adds an explicit taxonomic stratification pass and a phylogenetic-novelty pass that close those gaps.

## Methodology

Seven passes; each pick is tagged with the pass that selected it (`pass_origin` column) plus a per-pass reason.

1. **Phylum medoids** — one medoid per grower-bearing phylum (model with min sum-Jaccard reaction distance to its phylum-mates; sample capped at 50 per phylum). Guarantees every phylum is anchored.
2. **Reaction-coverage core** — greedy max-coverage on `seed.reaction` IDs, seeded with Pass 1. 12 picks.
3. **Taxonomic-novelty fill** — greedy farthest-point on lineage rank-distance (7=different superkingdom, 1=different species). Tie-break with Jaccard reaction distance. 12 picks; closes class/order gaps.
4. **Metabolite-coverage layer** — greedy max-coverage on cpd IDs, seeded with all prior picks. 8 picks.
5. **Constrained extremes** — 12 axis extremes + top-3 rare-reaction carriers, but skip any candidate whose class is already represented > 2.0× its expected share (walk down the ranking up to 20 before accepting). Cap 8.
6. **Hot-taxon medoids** — fills genus/family/order gaps that Pass 3 misses by sweeping the top grower-heavy missing taxa (threshold 20 growers) and adding their medoid. Cap 14; closes Pseudomonas, Burkholderia, etc. that sit inside already-anchored phyla.
7. **Farthest-point Jaccard fill** — fills the remaining slots by maximizing min reaction-set Jaccard distance to the panel. Pure diversity completion.

## Pass distribution

| Pass | Description | Picks |
|---|---|---|
| 1 | phylum medoids | 26 |
| 2 | reaction coverage | 8 |
| 3 | taxonomic novelty | 12 |
| 5 | constrained extremes | 8 |
| 6 | hot-taxon medoids | 14 |
| 7 | farthest-point Jaccard | 32 |

## Coverage achieved by the 100-model panel

- Reactions covered: **239 / 239** unique `seed.reaction` IDs (100.0%)
- Metabolites covered: **181 / 181** unique cpd IDs (100.0%)

## Taxonomic coverage

| Rank | distinct in panel | distinct among growers | coverage |
|---|---|---|---|
| superkingdom | 2 | 2 | 100.0% |
| phylum | 26 | 26 | 100.0% |
| class | 42 | 56 | 75.0% |
| order | 70 | 131 | 53.4% |
| family | 90 | 302 | 29.8% |
| genus | 97 | 870 | 11.1% |
| species | 100 | 2467 | 4.1% |

## Representativeness (Jaccard min-distance to nearest panel member)

| panel | median | mean | p90 | max |
|---|---|---|---|---|
| original | 0.192 | 0.181 | 0.255 | 0.304 |
| new | 0.149 | 0.147 | 0.237 | 0.309 |

## Gap closure vs original panel

### phylum (newly covered)

| phylum | grower count |
|---|---|
| Myxococcota | 21 |
| Gemmatimonadota | 3 |
| Rhodothermota | 3 |
| Balneolota | 1 |
| Ignavibacteriota | 1 |
| Calditrichota | 1 |

### class (newly covered)

| class | grower count |
|---|---|
| Myxococcia | 14 |
| Gemmatimonadia | 3 |
| Rhodothermia | 3 |
| Ignavibacteria | 1 |
| Blastocatellia | 1 |
| Calditrichia | 1 |
| Balneolia | 1 |
| Phycisphaerae | 1 |

### order (newly covered)

| order | grower count |
|---|---|
| Micrococcales | 122 |
| Rhodobacterales | 95 |
| Kitasatosporales | 70 |
| Vibrionales | 59 |
| Pseudonocardiales | 36 |
| Oceanospirillales | 35 |
| Synechococcales | 15 |
| Myxococcales | 14 |
| Deinococcales | 11 |
| Methylococcales | 5 |
| Gemmatimonadales | 3 |
| Rhodothermales | 3 |
| Planctomycetales | 3 |
| Ignavibacteriales | 1 |
| Calditrichales | 1 |

### genus (newly covered)

| genus | grower count |
|---|---|
| Pseudomonas | 84 |
| Streptomyces | 63 |
| Vibrio | 49 |
| Salmonella | 47 |
| Acinetobacter | 43 |
| Xanthomonas | 37 |
| Francisella | 35 |
| Alteromonas | 20 |
| Microbacterium | 18 |
| Amycolatopsis | 11 |
| Flavobacterium | 11 |
| Gordonia | 11 |
| Deinococcus | 11 |
| Novosphingobium | 6 |
| Synechococcus | 5 |

## Files

- `results/selected_ids_tax.txt` — 100 model IDs
- `results/selected_models_tax.csv` — per-model metrics, lineage, and selection reason
- `results/selected_models_tax.json` — same data + coverage stats
- `scripts/select_diverse_tax.py` — reproducible selection
- `notebooks/08_TaxonomyAwareSelection.ipynb` — interactive walkthrough

## Panel members (in selection order)

| # | model_id | pass | phylum | class | order | genus | reason |
|---|---|---|---|---|---|---|---|
| 1 | `GCF_000746525.1` | 1 | Pseudomonadota | Gammaproteobacteria | Pseudomonadales | Pseudomonas | phylum-medoid: Pseudomonadota (sum_jaccard=11.409) |
| 2 | `GCF_000024785.1` | 1 | Actinomycetota | Actinomycetes | Mycobacteriales | Gordonia | phylum-medoid: Actinomycetota (sum_jaccard=10.912) |
| 3 | `GCF_004006435.1` | 1 | Bacillota | Bacilli | Caryophanales | Bacillus | phylum-medoid: Bacillota (sum_jaccard=12.441) |
| 4 | `GCF_003076455.1` | 1 | Bacteroidota | Flavobacteriia | Flavobacteriales | Flavobacterium | phylum-medoid: Bacteroidota (sum_jaccard=11.569) |
| 5 | `GCF_000304375.1` | 1 | Campylobacterota | Epsilonproteobacteria | Campylobacterales | Campylobacter | phylum-medoid: Campylobacterota (sum_jaccard=10.497) |
| 6 | `GCF_000016865.1` | 1 | Unknown | Unknown | Unknown | Unknown | phylum-medoid: Unknown (sum_jaccard=13.373) |
| 7 | `GCF_000317045.1` | 1 | Cyanobacteriota | Cyanophyceae | Geitlerinematales | Geitlerinema | phylum-medoid: Cyanobacteriota (sum_jaccard=8.379) |
| 8 | `GCF_001278055.1` | 1 | Thermodesulfobacteriota | Desulfuromonadia | Desulfuromonadales | Desulfuromonas | phylum-medoid: Thermodesulfobacteriota (sum_jaccard=6.84) |
| 9 | `GCF_009017495.1` | 1 | Deinococcota | Deinococci | Deinococcales | Deinococcus | phylum-medoid: Deinococcota (sum_jaccard=4.052) |
| 10 | `GCF_000280925.3` | 1 | Myxococcota | Myxococcia | Myxococcales | Pseudomyxococcus | phylum-medoid: Myxococcota (sum_jaccard=3.925) |
| 11 | `GCF_009720525.1` | 1 | Planctomycetota | Planctomycetia | Planctomycetales | Gimesia | phylum-medoid: Planctomycetota (sum_jaccard=3.317) |
| 12 | `GCF_000178955.2` | 1 | Acidobacteriota | Terriglobia | Terriglobales | Granulicella | phylum-medoid: Acidobacteriota (sum_jaccard=2.377) |
| 13 | `GCF_000317895.1` | 1 | Bdellovibrionota | Bdellovibrionia | Bdellovibrionales | Bdellovibrio | phylum-medoid: Bdellovibrionota (sum_jaccard=0.981) |
| 14 | `GCF_001747405.1` | 1 | Chlorobiota | Chlorobiia | Chlorobiales | Chlorobaculum | phylum-medoid: Chlorobiota (sum_jaccard=1.175) |
| 15 | `GCF_000018865.1` | 1 | Chloroflexota | Chloroflexia | Chloroflexales | Chloroflexus | phylum-medoid: Chloroflexota (sum_jaccard=0.803) |
| 16 | `GCF_000218625.1` | 1 | Deferribacterota | Deferribacteres | Deferribacterales | Flexistipes | phylum-medoid: Deferribacterota (sum_jaccard=0.885) |
| 17 | `GCF_000017605.1` | 1 | Spirochaetota | Leptospiria | Leptospirales | Leptospira | phylum-medoid: Spirochaetota (sum_jaccard=1.133) |
| 18 | `GCF_002310495.1` | 1 | Verrucomicrobiota | Opitutia | Opitutales | Nibricoccus | phylum-medoid: Verrucomicrobiota (sum_jaccard=0.916) |
| 19 | `GCF_000695095.2` | 1 | Gemmatimonadota | Gemmatimonadia | Gemmatimonadales | Gemmatimonas | phylum-medoid: Gemmatimonadota (sum_jaccard=0.314) |
| 20 | `GCF_001518995.2` | 1 | Rhodothermota | Rhodothermia | Rhodothermales | Unknown | phylum-medoid: Rhodothermota (sum_jaccard=0.446) |
| 21 | `GCF_003353065.1` | 1 | Balneolota | Balneolia | Balneolales | Cyclonatronum | phylum-medoid: Balneolota (sum_jaccard=0.0) |
| 22 | `GCF_001886815.1` | 1 | Calditrichota | Calditrichia | Calditrichales | Caldithrix | phylum-medoid: Calditrichota (sum_jaccard=0.0) |
| 23 | `GCF_000253035.1` | 1 | Chlamydiota | Chlamydiia | Parachlamydiales | Parachlamydia | phylum-medoid: Chlamydiota (sum_jaccard=0.0) |
| 24 | `GCF_000177635.2` | 1 | Chrysiogenota | Chrysiogenia | Chrysiogenales | Desulfurispirillum | phylum-medoid: Chrysiogenota (sum_jaccard=0.0) |
| 25 | `GCF_000279145.1` | 1 | Ignavibacteriota | Ignavibacteria | Ignavibacteriales | Melioribacter | phylum-medoid: Ignavibacteriota (sum_jaccard=0.0) |
| 26 | `GCF_000284315.1` | 1 | Nitrospirota | Nitrospiria | Nitrospirales | Leptospirillum | phylum-medoid: Nitrospirota (sum_jaccard=0.0) |
| 27 | `GCF_000018625.1` | 2 | Pseudomonadota | Gammaproteobacteria | Enterobacterales | Salmonella | reaction-coverage rank 1: adds 5 previously-uncovered reactions |
| 28 | `GCF_000025265.1` | 2 | Actinomycetota | Thermoleophilia | Solirubrobacterales | Conexibacter | reaction-coverage rank 2: adds 3 previously-uncovered reactions |
| 29 | `GCF_001266795.1` | 2 | Pseudomonadota | Gammaproteobacteria | Pseudomonadales | Marinobacter | reaction-coverage rank 3: adds 2 previously-uncovered reactions |
| 30 | `GCF_009688965.1` | 2 | Thermodesulfobacteriota | Desulfobacteria | Desulfobacterales | Desulfosarcina | reaction-coverage rank 4: adds 2 previously-uncovered reactions |
| 31 | `GCF_000008525.1` | 2 | Campylobacterota | Epsilonproteobacteria | Campylobacterales | Helicobacter | reaction-coverage rank 5: adds 1 previously-uncovered reactions |
| 32 | `GCF_000015745.1` | 2 | Bacillota | Bacilli | Caryophanales | Geobacillus | reaction-coverage rank 6: adds 1 previously-uncovered reactions |
| 33 | `GCF_000016745.1` | 2 | Thermodesulfobacteriota | Desulfuromonadia | Geobacterales | Geotalea | reaction-coverage rank 7: adds 1 previously-uncovered reactions |
| 34 | `GCF_000756615.1` | 2 | Bacillota | Bacilli | Caryophanales | Paenibacillus | reaction-coverage rank 8: adds 1 previously-uncovered reactions |
| 35 | `GCF_000015445.1` | 3 | Pseudomonadota | Alphaproteobacteria | Hyphomicrobiales | Bartonella | taxonomic-novelty: rank-distance 5 (Pseudomonadota/Alphaproteobacteria/Hyphomicrobiales/Bartonella) |
| 36 | `GCF_001189515.2` | 3 | Actinomycetota | Coriobacteriia | Coriobacteriales | Olsenella | taxonomic-novelty: rank-distance 5 (Actinomycetota/Coriobacteriia/Coriobacteriales/Olsenella) |
| 37 | `GCF_002082765.1` | 3 | Bacillota | Negativicutes | Veillonellales | Veillonella | taxonomic-novelty: rank-distance 5 (Bacillota/Negativicutes/Veillonellales/Veillonella) |
| 38 | `GCF_001688905.2` | 3 | Pseudomonadota | Betaproteobacteria | Burkholderiales | Unknown | taxonomic-novelty: rank-distance 5 (Pseudomonadota/Betaproteobacteria/Burkholderiales/Unknown) |
| 39 | `GCF_000194135.1` | 3 | Campylobacterota | Desulfurellia | Desulfurellales | Hippea | taxonomic-novelty: rank-distance 5 (Campylobacterota/Desulfurellia/Desulfurellales/Hippea) |
| 40 | `GCF_000507245.1` | 3 | Spirochaetota | Spirochaetia | Spirochaetales | Salinispira | taxonomic-novelty: rank-distance 5 (Spirochaetota/Spirochaetia/Spirochaetales/Salinispira) |
| 41 | `GCF_001688725.2` | 3 | Bacteroidota | Bacteroidia | Bacteroidales | Bacteroides | taxonomic-novelty: rank-distance 5 (Bacteroidota/Bacteroidia/Bacteroidales/Bacteroides) |
| 42 | `GCF_000021485.1` | 3 | Pseudomonadota | Acidithiobacillia | Acidithiobacillales | Acidithiobacillus | taxonomic-novelty: rank-distance 5 (Pseudomonadota/Acidithiobacillia/Acidithiobacillales/Acidithiobacillus) |
| 43 | `GCF_002005145.1` | 3 | Bacillota | Clostridia | Eubacteriales | Desulforamulus | taxonomic-novelty: rank-distance 5 (Bacillota/Clostridia/Eubacteriales/Desulforamulus) |
| 44 | `GCF_000226295.1` | 3 | Acidobacteriota | Blastocatellia | Chloracidobacteriales | Chloracidobacterium | taxonomic-novelty: rank-distance 5 (Acidobacteriota/Blastocatellia/Chloracidobacteriales/Chloracidobacterium) |
| 45 | `GCF_001659705.1` | 3 | Bacteroidota | Chitinophagia | Chitinophagales | Arachidicoccus | taxonomic-novelty: rank-distance 5 (Bacteroidota/Chitinophagia/Chitinophagales/Arachidicoccus) |
| 46 | `GCF_000025945.1` | 3 | Thermodesulfobacteriota | Desulfobulbia | Desulfobulbales | Desulfotalea | taxonomic-novelty: rank-distance 5 (Thermodesulfobacteriota/Desulfobulbia/Desulfobulbales/Desulfotalea) |
| 47 | `GCF_003261575.2` | 5 | Pseudomonadota | Gammaproteobacteria | Enterobacterales | Klebsiella | extreme: largest by reactions |
| 48 | `GCF_000283635.1` | 5 | Bacillota | Bacilli | Lactobacillales | Streptococcus | extreme: smallest by reactions |
| 49 | `GCF_000632475.1` | 5 | Pseudomonadota | Alphaproteobacteria | Rhodospirillales | Azospirillum | extreme: largest by metabolites |
| 50 | `GCF_000014405.1` | 5 | Bacillota | Bacilli | Lactobacillales | Lactobacillus | extreme: smallest by metabolites |
| 51 | `GCF_000599545.1` | 5 | Actinomycetota | Actinomycetes | Mycobacteriales | Rhodococcus | extreme: largest by genes |
| 52 | `GCF_000014425.1` | 5 | Bacillota | Bacilli | Lactobacillales | Lactobacillus | extreme: smallest by genes |
| 53 | `GCF_000021045.1` | 5 | Pseudomonadota | Gammaproteobacteria | Pseudomonadales | Azotobacter | extreme: highest growth flux |
| 54 | `GCF_000195855.1` | 5 | Actinomycetota | Actinomycetes | Mycobacteriales | Mycobacterium | extreme: lowest growth flux (still growing) |
| 55 | `GCF_003991855.1` | 6 | Actinomycetota | Actinomycetes | Micrococcales | Microbacterium | hot-taxon medoid: order=Micrococcales (122 growers) |
| 56 | `GCF_001562115.1` | 6 | Pseudomonadota | Gammaproteobacteria | Alteromonadales | Alteromonas | hot-taxon medoid: order=Alteromonadales (102 growers) |
| 57 | `GCF_006351965.1` | 6 | Pseudomonadota | Alphaproteobacteria | Rhodobacterales | Oceanicola | hot-taxon medoid: order=Rhodobacterales (95 growers) |
| 58 | `GCF_000816885.1` | 6 | Pseudomonadota | Gammaproteobacteria | Lysobacterales | Xanthomonas | hot-taxon medoid: order=Lysobacterales (86 growers) |
| 59 | `GCF_000013325.1` | 6 | Pseudomonadota | Alphaproteobacteria | Sphingomonadales | Novosphingobium | hot-taxon medoid: order=Sphingomonadales (81 growers) |
| 60 | `GCF_001278075.1` | 6 | Actinomycetota | Actinomycetes | Kitasatosporales | Streptomyces | hot-taxon medoid: order=Kitasatosporales (70 growers) |
| 61 | `GCF_000018445.1` | 6 | Pseudomonadota | Gammaproteobacteria | Moraxellales | Acinetobacter | hot-taxon medoid: order=Moraxellales (62 growers) |
| 62 | `GCF_000039765.1` | 6 | Pseudomonadota | Gammaproteobacteria | Vibrionales | Vibrio | hot-taxon medoid: order=Vibrionales (59 growers) |
| 63 | `GCF_000016105.1` | 6 | Pseudomonadota | Gammaproteobacteria | Thiotrichales | Francisella | hot-taxon medoid: order=Thiotrichales (44 growers) |
| 64 | `GCF_000010945.1` | 6 | Pseudomonadota | Alphaproteobacteria | Acetobacterales | Acetobacter | hot-taxon medoid: order=Acetobacterales (38 growers) |
| 65 | `GCF_000009105.1` | 6 | Pseudomonadota | Betaproteobacteria | Neisseriales | Neisseria | hot-taxon medoid: order=Neisseriales (36 growers) |
| 66 | `GCF_009429145.1` | 6 | Actinomycetota | Actinomycetes | Pseudonocardiales | Amycolatopsis | hot-taxon medoid: order=Pseudonocardiales (36 growers) |
| 67 | `GCF_009846525.1` | 6 | Pseudomonadota | Gammaproteobacteria | Oceanospirillales | Vreelandella | hot-taxon medoid: order=Oceanospirillales (35 growers) |
| 68 | `GCF_003260975.1` | 6 | Bacteroidota | Cytophagia | Cytophagales | Echinicola | hot-taxon medoid: order=Cytophagales (34 growers) |
| 69 | `GCF_000270285.1` | 7 | Actinomycetota | Coriobacteriia | Eggerthellales | Eggerthella | farthest-point: min Jaccard distance to selected = 0.413 |
| 70 | `GCF_002127965.1` | 7 | Pseudomonadota | Betaproteobacteria | Burkholderiales | Oxalobacter | farthest-point: min Jaccard distance to selected = 0.395 |
| 71 | `GCF_003073475.1` | 7 | Actinomycetota | Actinomycetes | Actinomycetales | Actinobaculum | farthest-point: min Jaccard distance to selected = 0.383 |
| 72 | `GCF_900186975.1` | 7 | Actinomycetota | Actinomycetes | Propionibacteriales | Cutibacterium | farthest-point: min Jaccard distance to selected = 0.382 |
| 73 | `GCF_008805035.1` | 7 | Pseudomonadota | Betaproteobacteria | Neisseriales | Eikenella | farthest-point: min Jaccard distance to selected = 0.379 |
| 74 | `GCF_000747315.1` | 7 | Actinomycetota | Actinomycetes | Mycobacteriales | Corynebacterium | farthest-point: min Jaccard distance to selected = 0.370 |
| 75 | `GCF_001888165.1` | 7 | Thermodesulfobacteriota | Desulfuromonadia | Desulfuromonadales | Syntrophotalea | farthest-point: min Jaccard distance to selected = 0.370 |
| 76 | `GCF_001042635.1` | 7 | Actinomycetota | Actinomycetes | Bifidobacteriales | Bifidobacterium | farthest-point: min Jaccard distance to selected = 0.369 |
| 77 | `GCF_000284095.1` | 7 | Bacillota | Negativicutes | Selenomonadales | Pseudoselenomonas | farthest-point: min Jaccard distance to selected = 0.364 |
| 78 | `GCF_009649955.1` | 7 | Bacillota | Clostridia | Eubacteriales | Heliorestis | farthest-point: min Jaccard distance to selected = 0.355 |
| 79 | `GCF_000828835.1` | 7 | Pseudomonadota | Gammaproteobacteria | Thiotrichales | Thioploca | farthest-point: min Jaccard distance to selected = 0.353 |
| 80 | `GCF_000298115.2` | 7 | Bacillota | Bacilli | Lactobacillales | Lentilactobacillus | farthest-point: min Jaccard distance to selected = 0.353 |
| 81 | `GCF_000260965.1` | 7 | Pseudomonadota | Gammaproteobacteria | Thiotrichales | Methylophaga | farthest-point: min Jaccard distance to selected = 0.352 |
| 82 | `GCF_000307165.1` | 7 | Bacillota | Bacilli | Caryophanales | Amphibacillus | farthest-point: min Jaccard distance to selected = 0.346 |
| 83 | `GCF_001262075.1` | 7 | Pseudomonadota | Betaproteobacteria | Burkholderiales | Ottowia | farthest-point: min Jaccard distance to selected = 0.344 |
| 84 | `GCF_000241025.1` | 7 | Pseudomonadota | Gammaproteobacteria | Pasteurellales | Aggregatibacter | farthest-point: min Jaccard distance to selected = 0.343 |
| 85 | `GCF_000599985.1` | 7 | Pseudomonadota | Gammaproteobacteria | Orbales | Gilliamella | farthest-point: min Jaccard distance to selected = 0.343 |
| 86 | `GCF_000014585.1` | 7 | Cyanobacteriota | Cyanophyceae | Synechococcales | Synechococcus | farthest-point: min Jaccard distance to selected = 0.341 |
| 87 | `GCF_000743945.1` | 7 | Pseudomonadota | Betaproteobacteria | Burkholderiales | Basilea | farthest-point: min Jaccard distance to selected = 0.333 |
| 88 | `GCF_000025545.1` | 7 | Pseudomonadota | Gammaproteobacteria | Chromatiales | Thioalkalivibrio | farthest-point: min Jaccard distance to selected = 0.331 |
| 89 | `GCF_000148405.1` | 7 | Pseudomonadota | Gammaproteobacteria | Lysobacterales | Xylella | farthest-point: min Jaccard distance to selected = 0.331 |
| 90 | `GCF_002162355.1` | 7 | Bacillota | Bacilli | Caryophanales | Tumebacillus | farthest-point: min Jaccard distance to selected = 0.329 |
| 91 | `GCF_000092365.1` | 7 | Actinomycetota | Actinomycetes | Actinomycetales | Arcanobacterium | farthest-point: min Jaccard distance to selected = 0.328 |
| 92 | `GCF_000012805.1` | 7 | Pseudomonadota | Gammaproteobacteria | Chromatiales | Nitrosococcus | farthest-point: min Jaccard distance to selected = 0.326 |
| 93 | `GCF_002302395.1` | 7 | Bacteroidota | Flavobacteriia | Flavobacteriales | Capnocytophaga | farthest-point: min Jaccard distance to selected = 0.325 |
| 94 | `GCF_000233715.2` | 7 | Bacillota | Clostridia | Eubacteriales | Desulfoscipio | farthest-point: min Jaccard distance to selected = 0.324 |
| 95 | `GCF_002443115.1` | 7 | Actinomycetota | Actinomycetes | Micrococcales | Dermabacter | farthest-point: min Jaccard distance to selected = 0.314 |
| 96 | `GCF_000284115.1` | 7 | Planctomycetota | Phycisphaerae | Phycisphaerales | Phycisphaera | farthest-point: min Jaccard distance to selected = 0.314 |
| 97 | `GCF_001644685.1` | 7 | Pseudomonadota | Gammaproteobacteria | Methylococcales | Methylomonas | farthest-point: min Jaccard distance to selected = 0.313 |
| 98 | `GCF_000020525.1` | 7 | Chlorobiota | Chlorobiia | Chlorobiales | Chloroherpeton | farthest-point: min Jaccard distance to selected = 0.312 |
| 99 | `GCF_003261295.1` | 7 | Pseudomonadota | Betaproteobacteria | Burkholderiales | Polynucleobacter | farthest-point: min Jaccard distance to selected = 0.310 |
| 100 | `GCF_000145255.1` | 7 | Pseudomonadota | Betaproteobacteria | Nitrosomonadales | Gallionella | farthest-point: min Jaccard distance to selected = 0.309 |